<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/curated/006_Summary_from_Mydrive_Boltz2Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fetch from boltz2 folder prediction for affinity**

**Result saved to mydrive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import math
import pandas as pd
from pathlib import Path

# =========================
# Settings
# =========================
BASE_DIR = Path("/content/drive/MyDrive/Boltz2_Results")
OUT_XLSX = BASE_DIR / "Boltz2_PDB_summary.xlsx"

rows = []

# =========================
# Helper functions
# =========================
def read_json_safe(path):
    try:
        with open(path, "r") as f:
            return json.load(f), None
    except FileNotFoundError:
        return None, "missing file"
    except json.JSONDecodeError:
        return None, "bad json"
    except Exception as e:
        return None, str(e)

def convert_affinity_to_ic50_nM(affinity_pred_value):
    """
    Boltz-2 affinity_pred_value is log10(IC50 in uM).
    IC50_uM = 10 ** affinity_pred_value
    IC50_nM = IC50_uM * 1000
    """
    if affinity_pred_value is None:
        return None
    try:
        return (10 ** float(affinity_pred_value)) * 1000
    except Exception:
        return None

# =========================
# Scan all job folders
# =========================
for job_folder in sorted(BASE_DIR.iterdir()):
    if not job_folder.is_dir():
        continue

    jobname = job_folder.name
    pred_root = job_folder / "predictions"
    pred_folder = pred_root / jobname

    row = {
        "jobname": jobname,
        "affinity_pred_value": None,
        "Converted_IC50_nM": None,
        "affinity_probability_binary": None,
        "confidence_score": None,
        "complex_plddt": None,
        "status": "OK",
        "missing_or_error": ""
    }

    # Case 1: no predictions folder
    if not pred_root.exists():
        row["status"] = "BROKEN"
        row["missing_or_error"] = "missing predictions folder"
        rows.append(row)
        continue

    # Case 2: no predictions/jobname subfolder
    if not pred_folder.exists():
        row["status"] = "BROKEN"
        row["missing_or_error"] = f"missing predictions/{jobname} folder"
        rows.append(row)
        continue

    affinity_json = pred_folder / f"affinity_{jobname}.json"
    confidence_json = pred_folder / f"confidence_{jobname}_model_0.json"

    # Read affinity json
    affinity_data, affinity_err = read_json_safe(affinity_json)
    if affinity_data is not None:
        row["affinity_pred_value"] = affinity_data.get("affinity_pred_value")
        row["affinity_probability_binary"] = affinity_data.get("affinity_probability_binary")
        row["Converted_IC50_nM"] = convert_affinity_to_ic50_nM(row["affinity_pred_value"])
    else:
        row["status"] = "BROKEN"
        row["missing_or_error"] += f"affinity json: {affinity_err}; "

    # Read confidence json
    confidence_data, confidence_err = read_json_safe(confidence_json)
    if confidence_data is not None:
        row["confidence_score"] = confidence_data.get("confidence_score")

        # complex_plddt in json is usually 0-1 scale.
        # Convert to 0-100 scale for reporting.
        complex_plddt = confidence_data.get("complex_plddt")
        if complex_plddt is not None:
            row["complex_plddt"] = float(complex_plddt) * 100
    else:
        row["status"] = "BROKEN"
        row["missing_or_error"] += f"confidence json: {confidence_err}; "

    rows.append(row)

# =========================
# Create dataframe
# =========================
df = pd.DataFrame(rows)

# Sort by jobname, e.g. B0001, B0002...
df = df.sort_values("jobname").reset_index(drop=True)

# Optional rounding
df["affinity_pred_value"] = pd.to_numeric(df["affinity_pred_value"], errors="coerce").round(4)
df["Converted_IC50_nM"] = pd.to_numeric(df["Converted_IC50_nM"], errors="coerce").round(2)
df["affinity_probability_binary"] = pd.to_numeric(df["affinity_probability_binary"], errors="coerce").round(4)
df["confidence_score"] = pd.to_numeric(df["confidence_score"], errors="coerce").round(4)
df["complex_plddt"] = pd.to_numeric(df["complex_plddt"], errors="coerce").round(2)

# =========================
# Save to Excel
# =========================
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Boltz2_summary")

print(f"Saved summary to: {OUT_XLSX}")
print(f"Total jobs scanned: {len(df)}")
print(f"OK jobs: {(df['status'] == 'OK').sum()}")
print(f"Broken/incomplete jobs: {(df['status'] != 'OK').sum()}")

df.head()

Mounted at /content/drive
Saved summary to: /content/drive/MyDrive/Boltz2_Results/Boltz2_PDB_summary.xlsx
Total jobs scanned: 176
OK jobs: 175
Broken/incomplete jobs: 1


,jobname,affinity_pred_value,Converted_IC50_nM,affinity_probability_binary,confidence_score,complex_plddt,status,missing_or_error
0,PDBB_00001,-2.0209,9.53,0.9617,0.9588,95.37,OK,
1,PDBB_00002,-2.1778,6.64,0.9607,0.9541,94.82,OK,
2,PDBB_00003,0.1310,1352.10,0.7539,0.9669,96.20,OK,
3,PDBB_00004,-0.2053,623.34,0.8278,0.9694,96.45,OK,
4,PDBB_00005,-1.8510,14.09,0.9803,0.9759,97.26,OK,


**Search if have model_1**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd

# Main result folder
BASE_DIR = Path("/content/drive/MyDrive/Boltz2_Results")

# Output files
OUT_XLSX = BASE_DIR / "Boltz2_jobs_with_model_1.xlsx"
OUT_TXT = BASE_DIR / "Boltz2_jobs_with_model_1.txt"

rows = []

for job_folder in sorted(BASE_DIR.iterdir()):
    if not job_folder.is_dir():
        continue

    jobname = job_folder.name

    # Expected folder:
    # /MyDrive/Boltz2_Results/B0001/predictions/B0001/
    pred_folder = job_folder / "predictions" / jobname

    row = {
        "jobname": jobname,
        "has_model_1": False,
        "model_1_files": "",
        "status": "OK"
    }

    if not pred_folder.exists():
        row["status"] = f"missing predictions/{jobname} folder"
        rows.append(row)
        continue

    # Find files containing "_model_1"
    model_1_files = sorted([
        f.name for f in pred_folder.iterdir()
        if f.is_file() and "_model_1" in f.name
    ])

    if model_1_files:
        row["has_model_1"] = True
        row["model_1_files"] = "; ".join(model_1_files)

    rows.append(row)

df = pd.DataFrame(rows)

# Jobs that have model_1 files
df_model_1 = df[df["has_model_1"] == True].copy()

# Save Excel
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="all_jobs_checked")
    df_model_1.to_excel(writer, index=False, sheet_name="jobs_with_model_1")

# Save txt list of jobnames only
with open(OUT_TXT, "w") as f:
    for jobname in df_model_1["jobname"]:
        f.write(jobname + "\n")

print("Done.")
print(f"Total jobs checked: {len(df)}")
print(f"Jobs with model_1 files: {len(df_model_1)}")
print(f"Excel saved to: {OUT_XLSX}")
print(f"Jobname list saved to: {OUT_TXT}")

display(df_model_1[["jobname", "model_1_files"]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Done.
Total jobs checked: 96
Jobs with model_1 files: 91
Excel saved to: /content/drive/MyDrive/Boltz2_Results/Boltz2_jobs_with_model_1.xlsx
Jobname list saved to: /content/drive/MyDrive/Boltz2_Results/Boltz2_jobs_with_model_1.txt


,jobname,model_1_files
0,Plat0001,Plat0001_model_1.pdb; confidence_Plat0001_mode...
1,Plat0002,Plat0002_model_1.pdb; confidence_Plat0002_mode...
2,Plat0003,Plat0003_model_1.pdb; confidence_Plat0003_mode...
3,Plat0004,Plat0004_model_1.pdb; confidence_Plat0004_mode...
4,Plat0005,Plat0005_model_1.pdb; confidence_Plat0005_mode...
...,...,...
91,Plat0092,Plat0092_model_1.pdb; confidence_Plat0092_mode...
92,Plat0093,Plat0093_model_1.pdb; confidence_Plat0093_mode...
93,Plat0094,Plat0094_model_1.pdb; confidence_Plat0094_mode...
94,Plat0095,Plat0095_model_1.pdb; confidence_Plat0095_mode...
